In [1]:
import os
import glob
import pandas as pd

In [2]:
import os
import re

# Folder where your files are stored
folder = '/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data'

# Regex pattern:
# ^\d+_  → matches digits at the beginning followed by underscore
pattern = re.compile(r"^\d+_")

for filename in os.listdir(folder):
    old_path = os.path.join(folder, filename)

    # Skip folders
    if not os.path.isfile(old_path):
        continue

    # Apply renaming only if pattern matches
    if pattern.match(filename):
        new_filename = pattern.sub("", filename)  # Remove leading digits + underscore
        new_path = os.path.join(folder, new_filename)
        
        print(f"Renaming: {filename} → {new_filename}")
        os.rename(old_path, new_path)

print("Done!")


Done!


In [3]:
import pandas as pd
import os

def process_heart_rate_fullsegment(file_path, output_csv_name):
    print(f"Processing file: {file_path}...")

    # Detect header row (skip metadata)
    header_row_index = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if line.startswith("Row,Timestamp"):
                header_row_index = i
                break

    df = pd.read_csv(file_path, skiprows=header_row_index)
    df.columns = df.columns.str.strip()

    # Ensure Heart Rate column exists (with fallback)
    if 'Heart Rate' not in df.columns:
        print("Error: 'Heart Rate' column not found. Checking alternatives...")
        possible_cols = [c for c in df.columns if "heart" in c.lower()]
        if possible_cols:
            print(f"Using '{possible_cols[0]}' instead.")
            df.rename(columns={possible_cols[0]: 'Heart Rate'}, inplace=True)
        else:
            print("No usable Heart Rate column found.")
            return

    # Keep relevant columns
    df = df[['Timestamp', 'SlideEvent', 'SourceStimuliName', 'Heart Rate']].copy()
    df['Heart Rate'] = pd.to_numeric(df['Heart Rate'], errors='coerce')

    # Locate StartMedia and EndMedia
    start_indices = df.index[df['SlideEvent'] == 'StartMedia'].tolist()
    end_indices = df.index[df['SlideEvent'] == 'EndMedia'].tolist()

    if not start_indices or not end_indices:
        print("Error: Could not find StartMedia or EndMedia")
        return

    start_idx = start_indices[0]
    valid_end_indices = [i for i in end_indices if i > start_idx]

    if not valid_end_indices:
        print("Error: No EndMedia after StartMedia")
        return

    end_idx = valid_end_indices[0]

    # Extract full HR segment
    segment_df = df.loc[start_idx:end_idx].copy()

    # Add time elapsed
    start_time = segment_df['Timestamp'].iloc[0]
    segment_df['TimeElapsed'] = segment_df['Timestamp'] - start_time

    # Add metadata
    segment_df['Start_Index'] = start_idx
    segment_df['End_Index'] = end_idx
    segment_df['Start_Timestamp'] = segment_df['Timestamp'].iloc[0]
    segment_df['End_Timestamp'] = segment_df['Timestamp'].iloc[-1]

    # Output directory
    output_dir = "01_processed_data"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_csv_name)

    segment_df.to_csv(output_path, index=False)

    print(f"✔ Full HR segment saved to {output_path}")
    print(f"Rows exported: {len(segment_df)}")
    print(f"Start index: {start_idx} | End index: {end_idx}")

In [ ]:
import os
import glob

# --- Batch processing across participants ---

data_folder = "/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data"
result_folder = "/Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/01_processed_ECG"


os.makedirs(result_folder, exist_ok=True)

# Pattern: only CSV files in the ECG folder
pattern = os.path.join(data_folder, "*.csv")

for file_path in glob.glob(pattern):
    filename = os.path.basename(file_path)

    # Create output file name
    output_name = os.path.splitext(filename)[0] + '_HRresults.csv'
    output_path = os.path.join(result_folder, output_name)

    print(f"\n=== Running {filename} ===")
    process_heart_rate_fullsegment(file_path, output_path)



=== Running sub008_11-11-25 16h43m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data/sub008_11-11-25 16h43m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53761/3041276404.py:15: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full HR segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/01_processed_ECG/sub008_11-11-25 16h43m_HRresults.csv
Rows exported: 164125
Start index: 17 | End index: 164141

=== Running sub026_13-11-25 09h21m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data/sub026_13-11-25 09h21m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53761/3041276404.py:15: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full HR segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/01_processed_ECG/sub026_13-11-25 09h21m_HRresults.csv
Rows exported: 163994
Start index: 13 | End index: 164006

=== Running sub063_17-11-25 16h18m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data/sub063_17-11-25 16h18m.csv...


...

/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53761/3041276404.py:15: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full HR segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/01_processed_ECG/sub099_-11-25 09h04m_HRresults.csv
Rows exported: 163567
Start index: 15 | End index: 163581

=== Running sub185_04-12-25 12h19m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data/sub185_04-12-25 12h19m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53761/3041276404.py:15: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full HR segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/01_processed_ECG/sub185_04-12-25 12h19m_HRresults.csv
Rows exported: 163568
Start index: 47 | End index: 163614

=== Running sub027_13-11-25 09h39m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/01_ECG_data/sub027_13-11-25 09h39m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53761/3041276404.py:15: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)
